In [1]:
# ติดตั้งแพ็กเกจเมื่อรันใน repo — บน Kaggle มี pandas/matplotlib พร้อมอยู่แล้ว
from pathlib import Path

if Path("requirements-offline.txt").exists():
    get_ipython().run_line_magic(
        "pip", "install -r requirements-offline.txt --quiet")
else:
    print("ข้ามการติดตั้ง — ใช้แพ็กเกจที่มีใน environment (เช่น Kaggle)")

/Users/utarn/projects/datascience-metrology/.check-env/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# Module M2 — จัดการตารางด้วย pandas

|  |  |
|---|---|
| **เวลา** | Day 1, 12:45–14:00 (75 นาที) · Offline (CPU) |
| **ข้อมูล** | Gapminder · Coffee Quality Database |

**เมื่อจบโมดูลนี้ ผู้เรียนจะ...**

- โหลดไฟล์ CSV (และ Excel) เป็นตารางด้วย pandas
- เลือกคอลัมน์ สร้างคอลัมน์ใหม่ กรองแถว เรียงลำดับ — คำถาม "เอาเฉพาะที่" ตอบได้
- สรุปข้อมูลเป็นกลุ่มด้วย groupby — แทนการวน for ทีละกลุ่มจาก M1
- รวมหลายไฟล์เป็นตารางเดียวด้วย concat

**ข้อมูลที่ใช้ (citation):**

> Gapminder Foundation. *Life expectancy, GDP per capita & population*
> (excerpt 1952–2007). https://www.gapminder.org/free-material/ (CC BY 4.0)
>
> LeDoux, J. (2018). *coffee-quality-database* [Data set]. GitHub,
> https://github.com/jldbc/coffee-quality-database (data © Coffee Quality
> Institute)

## 1. ตารางใน Python ชื่อ DataFrame

โมดูลที่แล้วเราเก็บข้อมูลเองใน list/dict — ข้อมูลจริงมาเป็น**ไฟล์ตาราง**หลัก
พันถึงล้านแถว pandas คือเครื่องมือจัดการตารางของ Python แถว = บันทึก 1 รายการ,
คอลัมน์ = คุณสมบัติ 1 อย่าง (รูปร่างคุ้นตา — มันคือหน้าตา Excel นั่นเอง)

In [2]:
import tempfile

import pandas as pd
from pathlib import Path

import os


def data_root() -> Path:
    """หาโฟลเดอร์ datasets/ — ใช้ได้ทั้ง repo local, venv และ Kaggle.

    ลำดับการค้นหา:
    1. environment variable COURSE_DATA_ROOT (ชี้ตรงไปที่โฟลเดอร์ datasets)
    2. โฟลเดอร์ datasets/ ที่อยู่ระดับเดียวกับโน้ตบุ๊กหรือแม่ขึ้นไป (repo local)
    3. /kaggle/input — ทั้งกรณีอัปโหลด datasets/ ทั้งโฟลเดอร์
       หรืออัปโหลดแต่ละชุดแยกกัน (ชื่อโฟลเดอร์ต้องตรงกับใน datasets/)
    """
    env = os.environ.get("COURSE_DATA_ROOT")
    if env:
        return Path(env)
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").is_dir():
            return p / "datasets"
    kaggle = Path("/kaggle/input")
    if kaggle.is_dir():
        hits = sorted(kaggle.glob("**/datasets"))
        if hits:
            return hits[0]
        return kaggle
    raise SystemExit(
        "ไม่พบโฟลเดอร์ datasets/ — ตั้ง COURSE_DATA_ROOT "
        "หรือวาง datasets/ ไว้ข้างโน้ตบุ๊ก")


DATA = data_root()

In [3]:
gap = pd.read_csv(DATA / "gapminder" / "gapminderDataFiveYear.csv")
gap.head()

,country,year,pop,continent,lifeExp,gdpPercap
0,Afghanistan,1952,8425333.0,Asia,28.801,779.445314
1,Afghanistan,1957,9240934.0,Asia,30.332,820.853030
2,Afghanistan,1962,10267083.0,Asia,31.997,853.100710
3,Afghanistan,1967,11537966.0,Asia,34.020,836.197138
4,Afghanistan,1972,13079460.0,Asia,36.088,739.981106


In [4]:
# .head() แสดง 5 แถวแรก — ช็อคแรกของข้อมูลจริง: 1,704 แถว × 6 คอลัมน์
# .shape บอกมิติตาราง (แถว, คอลัมน์) · .info() สรุปชนิดข้อมูลแต่ละคอลัมน์
print(gap.shape)
gap.info()

(1704, 6)
<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   year       1704 non-null   int64  
 2   pop        1704 non-null   float64
 3   continent  1704 non-null   str    
 4   lifeExp    1704 non-null   float64
 5   gdpPercap  1704 non-null   float64
dtypes: float64(3), int64(1), str(2)
memory usage: 103.7 KB


## 2. เลือกคอลัมน์ — และสร้างคอลัมน์ใหม่

`df["col"]` ดึงคอลัมน์เดียว · `df[["a", "b"]]` ดึงหลายคอลัมน์ (สังเกต
**วงเล็บซ้อน** — ข้างในคือ list) และเขียนเพิ่มคอลัมน์ใหม่ได้เหมือนสร้างตัวแปร

In [5]:
print(gap["country"].head(3))    # คอลัมน์เดียว
gap[["country", "year", "lifeExp"]].head()    # หลายคอลัมน์

0    Afghanistan
1    Afghanistan
2    Afghanistan
Name: country, dtype: str


,country,year,lifeExp
0,Afghanistan,1952,28.801
1,Afghanistan,1957,30.332
2,Afghanistan,1962,31.997
3,Afghanistan,1967,34.020
4,Afghanistan,1972,36.088


In [6]:
# GDP รวมของประเทศ = GDP ต่อหัว × จำนวนประชากร — เก็บเป็นคอลัมน์ใหม่
gap["gdp_total"] = gap["gdpPercap"] * gap["pop"]
gap[["country", "year", "gdpPercap", "pop", "gdp_total"]].head()

,country,year,gdpPercap,pop,gdp_total
0,Afghanistan,1952,779.445314,8425333.0,6.567086e+09
1,Afghanistan,1957,820.853030,9240934.0,7.585449e+09
2,Afghanistan,1962,853.100710,10267083.0,8.758856e+09
3,Afghanistan,1967,836.197138,11537966.0,9.648014e+09
4,Afghanistan,1972,739.981106,13079460.0,9.678553e+09


## 3. กรองแถวด้วยเงื่อนไข — หัวใจของการวิเคราะห์

อยากได้ "เฉพาะปี 2007" "เฉพาะที่คาดหวังชีวิตเกิน 75" — สร้างเงื่อนไขให้ได้
`True`/`False` ต่อแถว (เหมือน `bool` จาก M1) แล้วใส่ในวงเล็บเหลี่ยม

In [7]:
mask_2007 = gap["year"] == 2007     # เงื่อนไข: True/False ต่อแถว
gap_2007 = gap[mask_2007]
print(len(gap_2007), "ประเทศในปี 2007")
gap_2007.head()

142 ประเทศในปี 2007


,country,year,pop,continent,lifeExp,gdpPercap,gdp_total
11,Afghanistan,2007,31889923.0,Asia,43.828,974.580338,3.107929e+10
23,Albania,2007,3600523.0,Europe,76.423,5937.029526,2.137641e+10
35,Algeria,2007,33333216.0,Africa,72.301,6223.367465,2.074449e+11
47,Angola,2007,12420476.0,Africa,42.731,4797.231267,5.958390e+10
59,Argentina,2007,40301927.0,Americas,75.320,12779.379640,5.150336e+11


In [8]:
# หลายเงื่อนไขพร้อมกัน: & (และ) · | (หรือ) — ต้องมีวงเล็บรอบแต่ละเงื่อนไข
rich_2007 = gap[(gap["year"] == 2007) & (gap["lifeExp"] > 75)]
print(len(rich_2007), "ประเทศ: ปี 2007 และคาดหวังชีวิตเกิน 75 ปี")

44 ประเทศ: ปี 2007 และคาดหวังชีวิตเกิน 75 ปี


## ✏️ ลองเอง 1 — นับประเทศที่อายุยืน

จาก `gap_2007` จงกรองเฉพาะประเทศที่มี `lifeExp` มากกว่า 80 ปี แล้วพิมพ์ว่ามี
ประเทศเหลืออยู่กี่ประเทศ

**คำใบ้:** สร้าง boolean mask จาก `gap_2007["lifeExp"]` แล้วใส่ในวงเล็บเหลี่ยม
นับผลลัพธ์ด้วย `len(...)` หรือ `.shape[0]`

In [ ]:
# ลองเขียนโค้ดของคุณที่นี่
# พิมพ์จำนวนประเทศที่ผ่านเงื่อนไข (ใช้ len หรือ .shape[0])

<div style="background-color:#e8f1fb; border-left:4px solid #2b6cb0; padding:10px 14px; border-radius:4px; color:#123d6b;">

**ผลลัพธ์ที่คาดหวัง**

<pre style="margin:8px 0 0 0; white-space:pre-wrap; color:#123d6b;">13</pre>

</div>

## 4. เรียงลำดับ — คำตอบของ "ใครมากสุด/น้อยสุด"

แถวในตารางมาแบบไหนก็ได้ `.sort_values()` จัดใหม่ตามคอลัมน์ที่เราสนใจ จับคู่
กับการกรองแล้ว คำถามแบบ "5 อันดับ" ตอบได้ทันที

In [9]:
rich_2007.sort_values(
    "gdpPercap", ascending=False
)[["country", "lifeExp", "gdpPercap"]].head()

,country,lifeExp,gdpPercap
1151,Norway,80.196,49357.19017
863,Kuwait,77.588,47306.98978
1367,Singapore,79.972,47143.17964
1619,United States,78.242,42951.65309
755,Ireland,78.885,40675.99635


## 5. สรุปเป็นกลุ่ม — groupby คือ for ที่เขียนแทนเรา

ใน M1 เราวน for รวมรายจ่ายเอง ถ้าอยากได้ "ค่าเฉลี่ยแยกตามทวีป" ต้องวนทวีป
ละรอบ groupby ทำสิ่งนั้นเป็นคำสั่งเดียว: แบ่งกลุ่ม → คำนวณ → รวมกลับ

In [10]:
by_continent = gap_2007.groupby("continent").agg(
    n_countries=("country", "size"),
    mean_life=("lifeExp", "mean"),
    mean_gdp=("gdpPercap", "mean"),
)
by_continent.sort_values("mean_life", ascending=False)

,n_countries,mean_life,mean_gdp
continent,,,
Oceania,2,80.719500,29810.188275
Europe,30,77.648600,25054.481636
Americas,25,73.608120,11003.031625
Asia,33,70.728485,12473.026870
Africa,52,54.806038,3089.032605


In [11]:
# แต่ละแถวของผลลัพธ์ = หนึ่งกลุ่ม — ตารางนี้คือ "รายงาน" ในตัว
# (ช่องว่างค่าเฉลี่ยระหว่างทวีปคือข้อค้นพลอยได้จากสี่บรรทัด)

## ✏️ ลองเอง 2 — ค่าเฉลี่ยอายุขัยแยกตามทวีป

จาก `gap_2007` จงหาค่าเฉลี่ย `lifeExp` ของแต่ละทวีป แล้วพิมพ์ผลลัพธ์ออกมา
(ผลลัพธ์มี 5 แถว ตั้งชื่อคอลัมน์ว่า `mean_life`)

**คำใบ้:** ใช้ `gap_2007.groupby("continent").agg(mean_life=("lifeExp", "mean"))`

In [ ]:
# ลองเขียนโค้ดของคุณที่นี่
# พิมพ์ตารางค่าเฉลี่ย lifeExp ของแต่ละทวีป (5 แถว)

<div style="background-color:#e8f1fb; border-left:4px solid #2b6cb0; padding:10px 14px; border-radius:4px; color:#123d6b;">

**ผลลัพธ์ที่คาดหวัง**

<pre style="margin:8px 0 0 0; white-space:pre-wrap; color:#123d6b;">           mean_life
continent           
Africa     54.806038
Americas   73.608120
Asia       70.728485
Europe     77.648600
Oceania    80.719500</pre>

</div>

## 6. Excel ก็อ่านได้ — ประตูเดียวกัน

งานในหน่วยงานส่วนใหญ่อยู่ในไฟล์ `.xlsx` pandas อ่าน/เขียนได้ผ่าน openpyxl
(ติดตั้งมาพร้อม env แล้ว) เราลองกับไฟล์ที่สร้างขึ้นชั่วคราว — จะได้เห็นทั้ง
ทางเข้าและทางออกของข้อมูล

In [12]:
# บันทึกผลการวิเคราะห์ออกเป็น Excel แล้วอ่านกลับเข้ามา
top10 = gap_2007.sort_values("lifeExp", ascending=False).head(10)
xlsx = Path(tempfile.mkdtemp()) / "top10_lifeExp.xlsx"
top10[["country", "lifeExp"]].to_excel(xlsx, index=False)
pd.read_excel(xlsx)

,country,lifeExp
0,Japan,82.603
1,"Hong Kong, China",82.208
2,Iceland,81.757
3,Switzerland,81.701
4,Australia,81.235
5,Spain,80.941
6,Sweden,80.884
7,Israel,80.745
8,France,80.657
9,Canada,80.653


## 7. รวมหลายไฟล์เป็นตารางเดียว — concat

ข้อมูลจริงมักถูกแบ่งเป็นหลายไฟล์ (ต่อปี ต่อเดือน ต่อภูมิภาค) การวิเคราะห์
ต้องประกอบก่อน `pd.concat()` วางตารางต่อกันแบบ "ต่อแถวท้ายตารางเดิม" —
ข้อมูลกาแฟของเรามีสองไฟล์: อาราบิกากับโรบัสตา

In [13]:
arabica = pd.read_csv(
    DATA / "coffee_quality" / "arabica_data_cleaned.csv")
robusta = pd.read_csv(
    DATA / "coffee_quality" / "robusta_data_cleaned.csv")
print("arabica:", arabica.shape, "· robusta:", robusta.shape)

arabica: (1311, 44) · robusta: (28, 44)


In [14]:
coffee = pd.concat([arabica, robusta], ignore_index=True)
print("รวมแล้ว:", coffee.shape)
coffee.groupby("Species").agg(
    n_reviews=("Species", "size"),
    mean_score=("Total.Cup.Points", "mean"),
)

รวมแล้ว: (1339, 49)


,n_reviews,mean_score
Species,,
Arabica,1311,82.115927
Robusta,28,80.868929


In [15]:
# แต่ทำไมตารางรวมมีช่องว่าง? — สองไฟล์มีคอลัมน์ไม่เหมือนกัน
# คอลัมน์ไหนอีกไฟล์ไม่มี จะกลายเป็นค่าว่าง (NaN)
# วิธีจัดการค่าว่าง = เรื่องแรกของโมดูลถัดไป
coffee[["Species", "Country.of.Origin", "Variety", "Altitude"]].head(8)

,Species,Country.of.Origin,Variety,Altitude
0,Arabica,Ethiopia,NaN,1950-2200
1,Arabica,Ethiopia,Other,1950-2200
2,Arabica,Guatemala,Bourbon,1600 - 1800 m
3,Arabica,Ethiopia,NaN,1800-2200
4,Arabica,Ethiopia,Other,1950-2200
5,Arabica,Brazil,NaN,NaN
6,Arabica,Peru,Other,NaN
7,Arabica,Ethiopia,NaN,1570-1700


## สรุป — ชุดเครื่องมือจัดการตาราง

| คำถาม | คำสั่ง |
|---|---|
| ตารางหน้าตาเป็นอย่างไร | `.head()` · `.shape` · `.info()` |
| เอาเฉพาะคอลัมน์ | `df["col"]` · `df[["a", "b"]]` |
| เอาเฉพาะบางแถว | กรอง boolean `df[mask]` ร่วมด้วย `&` / `\|` |
| จัดอันดับ | `.sort_values()` |
| สรุปต่อกลุ่ม | `.groupby()` + `.agg()` |
| รวมหลายไฟล์ | `pd.concat()` |
| Excel | `pd.read_excel()` / `.to_excel()` |

**ถัดไป (M3):** ตารางที่โหลดมายังไม่สะอาด — ค่า `?`, สะกดเพี้ยน, แถวซ้ำ,
ไฟล์ไทย encoding เก่า ทั้งหมดรออยู่